In [1]:
from itertools import product
from functools import reduce

import scipy as sc
from cirq_sic.wh import wh_povm
from cirq_sic.sics import load_sic_fiducial
from cirq_sic.utils import kron, rand_ket

import numpy as np
np.set_printoptions(precision=3, suppress=True)

X, Z = np.array([[0,1],[1,0]]), np.array([[1,0],[0,-1]])

In [2]:
def chsh_inequality():
    beta = 2
    v = np.array([-1,1])
    s = np.array([[1,1],[1,-1]])
    c = kron(s.flatten(), v, v)
    return (c, beta)

In [ ]:
class BellScenario:
    def __init__(self, reference_measurements, reference_states, observables, state):
        self.d = reference_measurements.shape[-1]
        self.n_parties = reference_measurements.shape[0]
        self.n_measurements = observables.shape[-1]

        self.reference_measurements = reference_measurements
        self.reference_states = reference_states
        self.observables = observables
        self.state = state

        self.P = np.array([[[(a@b).trace() for b in self.reference_states[i]] for a in self.reference_measurements[i]] for i in range(self.n_parties)]).real
        self.Phi = np.array([np.linalg.inv(_) for _ in self.P])
        self.R = np.array([kron(*[reference_measurements[i][j] for i, j in enumerate(idx)]) for idx in np.ndindex(*[self.d**2]*self.n_parties)])
        self.S = np.array([kron(*[reference_states[i][j] for i, j in enumerate(idx)]) for idx in np.ndindex(*[self.d**2]*self.n_parties)])
        self.PR = np.einsum("ijk, kj", self.R, state).real

        # shape: party, measurement, outcome
        outcomes = []
        projectors = []
        labels = []
        for i, agent_observables in enumerate(observables):
            outcomes_row = []
            projectors_row = []
            labels_row = []
            for j, O in enumerate(agent_observables):
                L, V = np.linalg.eigh(O)
                Pi = np.array([np.outer(v, v.conj()) for v in V.T])
                outcomes_row.append(L)
                projectors_row.append(Pi)
                labels_row.append([f"A{i}: o{k} of O{j}" for k in range(len(L))])
            outcomes.append(outcomes_row)
            projectors.append(projectors_row)
            labels.append(labels_row)
        self.outcomes = np.array(outcomes)
        self.projectors = np.array(projectors)
        self.labels = np.array(labels)

        self.tensor_projectors = np.array([reduce(np.kron, (proj[idx] for proj, idx in zip(projectors, outcome_indices)))\
                                    for outcome_indices in product(*[range(n) for n in [self.d]*self.n_parties])])\
                                        .reshape(-1, self.d**self.n_parties, self.d**self.n_parties)
        self.p = np.einsum("ijk, kj", self.tensor_projectors, state)

        label_blocks = []
        for meas_choice in product(range(self.n_measurements), repeat=self.n_parties):
            outcome_labels = self.labels[np.arange(self.n_parties), meas_choice]
            block = [' ⊗ '.join(outcome_labels[p][outcome_choice[p]] for p in range(self.n_parties))
                for outcome_choice in product(range(d), repeat=self.n_parties)]
            label_blocks.append(block)
        self.tensor_labels = np.array(label_blocks).reshape(-1)
    
        PER = np.array([np.einsum("ijkl, mlk", self.projectors[i], reference_states[i]) for i in range(self.n_parties)]).real
        JPER = np.array([reduce(np.kron, (per[idx] for per, idx in zip(PER, outcome_indices)))\
                            for outcome_indices in product(*[range(n) for n in [self.d]*self.n_parties])])\
                                .reshape(-1, self.d**(2*self.n_parties))
        self.phi = kron(*self.Phi).flatten()
        self.T = kron(JPER, self.PR).reshape(-1, self.phi.shape[0])
        if not np.allclose(self.T @ self.phi, self.p):
            print("Warning: T phi != p")
            print(f"\tdistance: {np.linalg.norm(self.T @ self.phi - self.p)}")
            print(f"\tPdets: {np.linalg.det(self.P)}")

        self.T_singular_values = np.linalg.svd(self.T, compute_uv=False)

    def sigma_max(self):
        return self.T_singular_values[0]
    
    def bound(self, inequality):
        c, beta = inequality
        Delta = c @ self.p - beta
        return Delta/(self.sigma_max()*np.linalg.norm(c))

In [4]:
d = 2
n_measurements = 2
n_parties = 2

observables = np.array([[Z, X], [(Z+X)/np.sqrt(2), (Z-X)/np.sqrt(2)]])
ket = np.array([1,0,0,1])/np.sqrt(2)
state = np.outer(ket, ket.conj())

reference_measurements = np.array([wh_povm(load_sic_fiducial(d)) for i in range(n_parties)])
reference_states = np.array([d*reference_measurements[i] for i in range(n_parties)])
scenario = BellScenario(reference_measurements, reference_states, observables, state)
sic_sigma_max = scenario.sigma_max(); 
sic_sigma_max

np.float64(1.154700538379251)

In [14]:
def test_sigma_max(f, N=1000, same=False):
    sigma_max_values = []
    for t in range(N):
        if same:
            R = f()
            random_measurements = np.array([R for i in range(n_parties)])
        else:
            random_measurements = np.array([f() for i in range(n_parties)])
        random_states = np.array([[m/m.trace() for m in M] for M in random_measurements])
        random_scenario = BellScenario(random_measurements, random_states, observables, state)
        sigma_max_values.append(random_scenario.sigma_max())
    sigma_max_values = np.array(sigma_max_values) 
    violations = sigma_max_values <= sic_sigma_max
    return sigma_max_values[violations]

In [15]:
def rand_rank1_povm(d, n):
    return np.array([np.outer(r,r.conj()) for r in sc.stats.unitary_group.rvs(n)[:, :d]])

print(test_sigma_max(lambda : rand_rank1_povm(d, d**2), same=False))
print(test_sigma_max(lambda : rand_rank1_povm(d, d**2), same=True))

	distance: 3.5484622254252165e-06
	Pdets: [0. 0.]
[1.15  1.12  1.152 1.15  1.111 1.135 1.149 1.128]
	distance: 2.6594566232277435e-05
	Pdets: [0. 0.]
	distance: 0.01100028122161038
	Pdets: [0. 0.]
	distance: 4.113052222503354e-05
	Pdets: [0. 0.]
	distance: 0.0007845046925281875
	Pdets: [0. 0.]
[1.141 1.119 1.143 1.125 1.152 1.149 1.145 1.151 1.146 1.129 1.126 1.1
 1.153 1.137 1.136 1.143 1.129 1.142 1.142 1.081 1.12  1.12  1.144 1.151
 1.115 1.113 1.153 1.149 1.142]


In [16]:
def rand_rank1_wh_povm(d):
    return wh_povm(rand_ket(d))

print(test_sigma_max(lambda : rand_rank1_wh_povm(d), same=False))
print(test_sigma_max(lambda : rand_rank1_wh_povm(d), same=True))

[1.153 1.054 1.06  1.098 1.103 1.139 1.143 1.15  1.044 1.104 1.131 1.092
 1.046 1.006 1.054 1.067 1.146 1.084 1.044 1.119 1.152 1.107 1.134 1.101
 1.115 1.134 1.09  1.083 1.066 1.129 1.001 1.136 1.101 1.064 1.113 1.098
 1.062 1.078 1.095 1.09  1.135 1.037 1.013 1.01  1.105 1.039 1.118 1.044
 1.06  1.09  1.118 1.026 1.068 1.111 1.139 1.084 1.051 1.045 1.042 1.146
 1.083 1.119 1.097 1.148 1.063 1.071 1.14  1.052 1.125 1.012 1.071 1.117
 1.1   1.08  1.026 1.029 1.091 1.024 1.146 1.108 1.102 1.107 1.052 1.053
 1.057 1.106 1.066 1.065 1.046 1.111 1.013 1.098 1.128 1.105 1.14  1.139
 1.057 1.103 1.146 1.135 1.142 1.126 1.09  1.15  1.009 1.133 1.068 1.06
 1.115 1.126 1.066 1.135 1.074 1.104 1.14  1.122 1.025 1.095 1.151 1.13
 1.031 1.003 1.004 1.153 1.084 1.133 1.138 1.083 1.122 1.12  1.058 1.096
 1.045 1.123 1.044 1.118 1.148 1.131 1.113 1.141 1.108 1.085 1.056 1.092
 1.125 1.143 1.149 1.12  1.136 1.017 1.115 1.118 1.14  1.096 1.146 1.148
 1.132 1.006 1.146 1.132 1.154 1.123 1.128 1.053 1.09

In [17]:
def rand_unbiased_rank1_povm(d, n, rtol=1e-8, atol=1e-8):
    R = np.random.randn(d, n) + 1j*np.random.randn(d, n) 
    while not (np.allclose(R @ R.conj().T, (n/d)*np.eye(d), rtol=rtol, atol=atol) and\
               np.allclose(np.linalg.norm(R, axis=0), np.ones(n), rtol=rtol, atol=atol)):
        R = sc.linalg.polar(R)[0]
        R = np.array([state/np.linalg.norm(state) for state in R.T]).T
    R = sc.linalg.polar(R)[0]
    return np.array([np.outer(r,r.conj()) for r in R.T])

print(test_sigma_max(lambda : rand_unbiased_rank1_povm(d, d**2), same=False))
print(test_sigma_max(lambda : rand_unbiased_rank1_povm(d, d**2), same=True))

[1.04  1.15  1.079 1.064 1.114 1.154 1.126 1.144 1.14  1.038 1.129 1.119
 1.148 1.108 1.123 1.146 1.094 1.142 1.101 1.151 1.123 1.135 1.147 1.111
 1.152 1.124 1.124 1.123 1.12  1.147 1.089 1.143 1.114 1.095 1.141 1.139
 1.136 1.155 1.131 1.113 1.149 1.154 1.107 1.138 1.116 1.15  1.109 1.154
 1.15  1.122 1.108 1.154 1.049 1.146 1.145 1.142 1.074 1.146 1.154 1.154
 1.115 1.052 1.149 1.136 1.085 1.1   1.141 1.115 1.094 1.145 1.13  1.148
 1.077 1.147 1.151 1.144 1.131 1.11  1.08  1.15  1.128 1.136 1.098 1.137
 1.13  1.05  1.117 1.151 1.1   1.111 1.149 1.144 1.139 1.143 1.152 1.146
 1.095 1.138 1.15  1.122 1.085 1.131 1.118 1.131 1.13  1.06  1.115 1.13
 1.11  1.119 1.151 1.127 1.149 1.124 1.145 1.08  1.143 1.133 1.149 1.125
 1.141 1.15  1.112 1.107 1.141 1.13  1.131 1.108 1.12  1.064 1.107 1.134
 1.118 1.096 1.071 1.097 1.122 1.069 1.124 1.147 1.134 1.127 1.143 1.12
 1.15  1.148 1.154 1.134 1.15  1.092 1.152 1.123 1.144 1.078 1.065 1.134
 1.133 1.146 1.144 1.123 1.075 1.131 1.145 1.122 1.12

In [18]:
def rand_povm(d, n):
    U =  sc.stats.unitary_group.rvs(d*n)[:, :d]
    K = U.reshape(n, d, d)
    R = np.array([k.conj().T @ k for k in K])
    return R

print(test_sigma_max(lambda : rand_povm(d, d**2), same=False))
print(test_sigma_max(lambda : rand_povm(d, d**2), same=True))

[1.103 1.133 1.096 1.122 1.125 1.139 1.129 1.1   1.12  1.112 1.119 1.141
 1.118 1.138 1.147 1.126 1.123 1.089 1.044 1.125 1.154 1.118 1.052 1.146
 1.093 1.112 1.096 1.075 1.122 1.122 1.11  1.107 1.13  1.091 1.132 1.093
 1.082 1.105 1.127 1.152 1.116 1.133 1.148 1.151 1.12  1.094 1.138 1.121
 1.117 1.119 1.119 1.154 1.118 1.118 1.151 1.121 1.088 1.148 1.119 1.099
 1.137 1.121 1.124 1.144 1.136 1.118 1.13  1.11  1.12  1.118 1.115 1.034
 1.094 1.09  1.103 1.117 1.115 1.142 1.077 1.129 1.133 1.147 1.149 1.069
 1.144 1.137 1.145 1.145 1.142 1.081 1.084 1.096 1.116 1.137 1.116 1.079
 1.112 1.074 1.132 1.109 1.123 1.135 1.143 1.127 1.101 1.111 1.106 1.099
 1.15  1.063 1.108 1.151 1.038 1.112 1.125 1.099 1.102 1.1   1.098 1.136
 1.132 1.103 1.15  1.115 1.089 1.126 1.122 1.144 1.153 1.151 1.125 1.133
 1.146 1.119 1.138 1.115 1.14  1.153 1.09  1.153 1.154 1.061 1.087 1.128
 1.11  1.107 1.105 1.117 1.127 1.153 1.136 1.138 1.103 1.088 1.131 1.135
 1.099 1.058 1.111 1.138 1.077 1.122 1.074 1.154 1.